In [4]:
with open("ai-prs-from-repos-with-human.txt", "r", encoding="utf-8") as fin, open("ai-prs-from-repos-with-human-without-repo-url.txt", "w", encoding="utf-8") as fout:
    next(fin)

    for line in fin:
        cols = line.strip().split("\t")
        if len(cols) >= 3:
            fout.write(f"{cols[0]}\t{cols[2]}\n")
        else:
            fout.write(line)


In [6]:
import os
import requests
import re
import json
import time
from datetime import datetime

with open(".env", "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("GITHUB_TOKEN="):
            TOKEN = line.strip().split("=", 1)[1].strip().strip("'").strip('"')
            break

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json"
}

def parse_pr_link(link):
    pattern = r"https://github\.com/([^/]+)/([^/]+)/pull/(\d+)"
    match = re.match(pattern, link.strip())
    if match:
        owner, repo, pr_number = match.groups()
        return owner, repo, int(pr_number)
    return None, None, None

def handle_rate_limit(response):
    if response.status_code == 403 and "X-RateLimit-Remaining" in response.headers:
        remaining = int(response.headers.get("X-RateLimit-Remaining", 0))
        reset_ts = int(response.headers.get("X-RateLimit-Reset", 0))
        now_ts = int(time.time())

        if remaining == 0:
            wait_time = reset_ts - now_ts + 5
            reset_time = datetime.utcfromtimestamp(reset_ts).strftime('%Y-%m-%d %H:%M:%S')
            print(f"Limit API osiągnięty. Czekam {wait_time} sek. (reset o {reset_time} UTC)...")
            time.sleep(max(wait_time, 0))
            return True
    return False

with open("ai-prs-from-repos-with-human-without-repo-url.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

output_path = "pr_lines_ai_pr_from_repo_with_human_summary.txt"
batch_size = 5000
total_lines = len(lines)

print(f"Znaleziono {total_lines} linków PR. Przetwarzanie po {batch_size} na partię...")

with open(output_path, "a", encoding="utf-8") as out:
    for i in range(0, total_lines, batch_size):
        batch = lines[i:i + batch_size]
        print(f"\nPrzetwarzanie partii {i // batch_size + 1} ({len(batch)} linków)...")

        for line in batch:
            parts = line.split()
            if len(parts) != 2:
                print(f"Niepoprawny format: {line}")
                continue

            pr_id, link = parts
            owner, repo, pr_number = parse_pr_link(link)
            if not owner:
                print(f"Niepoprawny link: {link}")
                continue

            url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}"

            # Obsluga limitow
            while True:
                response = requests.get(url, headers=headers)
                if handle_rate_limit(response):
                    continue

                if response.status_code != 200:
                    print(f"Błąd {response.status_code} przy pobieraniu: {url}")
                    break

                pr_data = response.json()
                additions = pr_data.get("additions", 0)
                deletions = pr_data.get("deletions", 0)
                changed_lines = additions + deletions

                record = {
                    "pr_id": int(pr_id),
                    "pr_link": link,
                    "additions": int(additions),
                    "deletions": int(deletions),
                    "changed_lines": int(changed_lines)
                }

                out.write(json.dumps(record, ensure_ascii=False) + "\n")
                break  # wyjscie z petli while True

        if i + batch_size < total_lines:
            print(f"Krótka przerwa 10 sekund przed kolejną partią ({i + batch_size}/{total_lines})...")
            time.sleep(10)

print(f"\nZakończono. Dane zapisano do pliku: {output_path}")

Znaleziono 9797 linków PR. Przetwarzanie po 5000 na partię...

Przetwarzanie partii 1 (5000 linków)...


/tmp/ipykernel_1081173/1469180229.py:35: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  reset_time = datetime.utcfromtimestamp(reset_ts).strftime('%Y-%m-%d %H:%M:%S')


Limit API osiągnięty. Czekam 718 sek. (reset o 2025-10-27 08:05:32 UTC)...
Krótka przerwa 10 sekund przed kolejną partią (5000/9797)...

Przetwarzanie partii 2 (4797 linków)...
Limit API osiągnięty. Czekam 935 sek. (reset o 2025-10-27 09:05:37 UTC)...

Zakończono. Dane zapisano do pliku: pr_lines_ai_pr_from_repo_with_human_summary.txt


In [7]:
# Konwersja na parquet
import pandas as pd
import json

input_file = "pr_lines_ai_pr_from_repo_with_human_summary.txt"
output_file = "ai_pr_from_repo_with_human_lines_summary.parquet"

records = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line.strip()))

df = pd.DataFrame(records)
df.to_parquet(output_file, index=False)

print(f"Zapisano dane do pliku Parquet: {output_file}")
print(f"Liczba rekordów: {len(df)}")

Zapisano dane do pliku Parquet: ai_pr_from_repo_with_human_lines_summary.parquet
Liczba rekordów: 9797
